<a href="https://colab.research.google.com/github/VintaBytes/Ciencia-de-datos/blob/main/libros/ciencia-de-datos-con-python/volumen-01/cuadernos/capitulo-017-limpieza-de-textos-y-categorias.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Capítulo 17 — Limpieza de textos y categorías

## Breve repaso

En el trabajo anterior sobre duplicados vimos que una parte importante de la limpieza de datos consiste en interpretar los problemas antes de resolverlos. No alcanza con detectar filas repetidas: también necesitamos comprender qué representa cada fila, qué criterio usamos para definir un duplicado y qué información podríamos perder al eliminar registros.

En este capítulo vamos a trabajar con otro problema muy frecuente en la preparación de datos: la inconsistencia en columnas de texto y categorías.

Muchas columnas categóricas parecen simples al principio. Por ejemplo, una columna puede indicar el producto vendido, el método de pago o la ubicación de una venta. Sin embargo, cuando los valores fueron cargados manualmente, combinados desde distintas fuentes o exportados desde sistemas diferentes, pueden aparecer pequeñas diferencias de escritura que afectan el análisis.

Para una persona, valores como `"Coffee"`, `" coffee"` y `"COFFEE"` pueden representar el mismo producto. Para Pandas, en cambio, son valores distintos. Lo mismo puede ocurrir con `"Credit Card"`, `"credit card"`, `"Credito"` y `"Crédito"`, o con variantes como `"Takeaway"`, `"Take away"` y `"take-away"`.

Estas diferencias no son solo un problema estético. Si las categorías están escritas de manera inconsistente, los conteos se fragmentan, los filtros pueden dejar registros afuera, los agrupamientos pueden generar resultados incorrectos y las conclusiones pueden ser engañosas.

Por eso, limpiar textos y categorías es una parte importante de la calidad de los datos. En este capítulo vamos a revisar problemas como espacios innecesarios, diferencias entre mayúsculas y minúsculas, tildes, variantes de escritura, valores desconocidos y valores que indican error.

Al finalizar este capítulo deberías poder:

- Comprender por qué las inconsistencias de texto afectan el análisis.
- Revisar valores únicos y frecuencias en columnas categóricas.
- Detectar diferencias causadas por espacios, mayúsculas o minúsculas.
- Usar métodos de texto como `.str.strip()` y `.str.lower()`.
- Reemplazar valores inconsistentes usando `replace()`.
- Decidir cómo tratar valores como `"UNKNOWN"`, `"ERROR"` o valores faltantes reales.
- Crear versiones limpias de columnas categóricas.
- Verificar que una limpieza de categorías haya funcionado.

## Dataset de trabajo

Para estudiar limpieza de textos y categorías vamos a usar un dataset sintético de ventas de cafetería.

A diferencia del dataset real de Kaggle, este dataset fue construido especialmente para mostrar problemas frecuentes en columnas categóricas. Incluye valores con espacios al comienzo o al final, diferencias entre mayúsculas y minúsculas, variantes de escritura, valores desconocidos, valores de error y datos faltantes reales.

Usar un dataset pequeño y controlado nos permite observar con claridad el problema. La idea no es simular todos los detalles de un archivo real, sino construir un ejemplo didáctico que nos permita entender por qué estas inconsistencias afectan el análisis.

In [101]:
import pandas as pd

datos = {
    "id_venta": [
        "V001", "V002", "V003", "V004", "V005",
        "V006", "V007", "V008", "V009", "V010",
        "V011", "V012", "V013", "V014", "V015"
    ],
    "producto": [
        "Coffee", " coffee", "COFFEE", "Tea", "tea ",
        "Sandwich", "sandwich", "Cake", "cake ", "Juice",
        "UNKNOWN", "ERROR", None, "Smoothie", "smoothie "
    ],
    "metodo_pago": [
        "Cash", "cash", " CASH ", "Credit Card", "credit card",
        "Credito", "Crédito", "Digital Wallet", "digital wallet", "wallet",
        "UNKNOWN", "ERROR", None, "Debit Card", "debit card "
    ],
    "ubicacion": [
        "In-store", "in-store", " In-store ", "Takeaway", "takeaway",
        "TAKEAWAY", "Delivery", "delivery ", "DELIVERY", "In store",
        "UNKNOWN", "ERROR", None, "Take away", "take-away"
    ],
    "cantidad": [
        2, 1, 3, 2, 1,
        1, 2, 1, 3, 2,
        1, 1, 2, 1, 2
    ],
    "precio_unitario": [
        1200, 1200, 1200, 900, 900,
        2500, 2500, 1800, 1800, 1500,
        0, 0, 1200, 2200, 2200
    ]
}

df = pd.DataFrame(datos)

df

,id_venta,producto,metodo_pago,ubicacion,cantidad,precio_unitario
0,V001,Coffee,Cash,In-store,2,1200
1,V002,coffee,cash,in-store,1,1200
2,V003,COFFEE,CASH,In-store,3,1200
3,V004,Tea,Credit Card,Takeaway,2,900
4,V005,tea,credit card,takeaway,1,900
5,V006,Sandwich,Credito,TAKEAWAY,1,2500
6,V007,sandwich,Crédito,Delivery,2,2500
7,V008,Cake,Digital Wallet,delivery,1,1800
8,V009,cake,digital wallet,DELIVERY,3,1800
9,V010,Juice,wallet,In store,2,1500


Al observar el `DataFrame`, algunos problemas se ven rápidamente. Por ejemplo, en la columna `producto` aparecen valores como `"Coffee"`, `" coffee"` y `"COFFEE"`. Aunque probablemente representan el mismo producto, Pandas los interpreta como valores diferentes.

También aparecen valores como `"UNKNOWN"`, `"ERROR"` y `None`. Estos casos requieren una decisión distinta. No son simplemente variantes de escritura de un producto, sino señales de información desconocida, errores de carga o datos faltantes.

En este capítulo vamos a trabajar paso a paso para detectar, limpiar y verificar este tipo de problemas.

## Revisar valores únicos y frecuencias

Antes de limpiar una columna de texto, necesitamos revisar qué valores contiene.

Una forma directa de hacerlo es usar `value_counts()`. Esta herramienta permite contar cuántas veces aparece cada valor dentro de una columna.

Vamos a empezar con la columna `producto`.

In [102]:
df["producto"].value_counts(dropna=False)

,count
producto,
Coffee,1
coffee,1
COFFEE,1
Tea,1
tea,1
Sandwich,1
sandwich,1
Cake,1
cake,1


La salida muestra un problema importante: Pandas cuenta como productos diferentes valores que, para una persona, probablemente representan lo mismo.

Por ejemplo:

```text
Coffee
 coffee
COFFEE
```

aparecen como valores separados. Lo mismo ocurre con:

```text
Tea
tea
```

o con:

```text
Cake
cake
```

Esto ocurre porque Pandas compara los textos de manera exacta. Si un valor tiene un espacio al comienzo, un espacio al final o una diferencia de mayúsculas, Pandas lo considera distinto.

Este problema afecta directamente el análisis. Si queremos saber cuántas ventas hubo de café, no obtendremos un único conteo para `"Coffee"`, sino varios conteos separados para cada variante de escritura.

También podemos revisar otras columnas categóricas.

In [103]:
df["metodo_pago"].value_counts(dropna=False)

,count
metodo_pago,
Cash,1
cash,1
CASH,1
Credit Card,1
credit card,1
Credito,1
Crédito,1
Digital Wallet,1
digital wallet,1


In [104]:
df["ubicacion"].value_counts(dropna=False)

,count
ubicacion,
In-store,1
in-store,1
In-store,1
Takeaway,1
takeaway,1
TAKEAWAY,1
Delivery,1
delivery,1
DELIVERY,1


Estas salidas muestran que el problema no aparece en una sola columna. Hay inconsistencias en productos, métodos de pago y ubicaciones.

Por eso, antes de analizar datos categóricos, conviene revisar sus valores únicos y frecuencias. Si no hacemos esta revisión, podríamos construir filtros, conteos o agrupamientos sobre categorías fragmentadas.

## Espacios innecesarios al comienzo o al final

Uno de los problemas más comunes en columnas de texto es la presencia de espacios innecesarios.

Un valor como `"Coffee"` no es igual a `" coffee"` para Pandas, porque el segundo tiene un espacio al comienzo. Lo mismo ocurre con `"tea "` o `"cake "`, que tienen un espacio al final.

Estos espacios pueden ser difíciles de ver a simple vista, pero afectan conteos, filtros y agrupamientos. Por ejemplo, si filtramos las filas donde el producto es exactamente `"Coffee"`, no vamos a incluir automáticamente las filas donde el producto aparece como `" coffee"` o `"COFFEE"`.

Podemos observar algunos valores de la columna `producto` para notar estas diferencias.

In [105]:
df["producto"].unique()

array(['Coffee', ' coffee', 'COFFEE', 'Tea', 'tea ', 'Sandwich',
       'sandwich', 'Cake', 'cake ', 'Juice', 'UNKNOWN', 'ERROR', None,
       'Smoothie', 'smoothie '], dtype=object)

Para quitar espacios al comienzo y al final de cada texto, usamos `.str.strip()`.

En lugar de modificar directamente la columna original, vamos a crear una nueva columna llamada `producto_limpio`. Esto nos permitirá comparar los valores originales con los valores transformados.

In [106]:
df["producto_limpio"] = df["producto"].str.strip()

df[["producto", "producto_limpio"]]

,producto,producto_limpio
0,Coffee,Coffee
1,coffee,coffee
2,COFFEE,COFFEE
3,Tea,Tea
4,tea,tea
5,Sandwich,Sandwich
6,sandwich,sandwich
7,Cake,Cake
8,cake,cake
9,Juice,Juice


Ahora los valores que tenían espacios al comienzo o al final quedaron corregidos en `producto_limpio`.

Observemos también que los valores faltantes reales no se transformaron en texto. Pandas conserva esos casos como valores ausentes, de modo que la limpieza de texto no reemplaza automáticamente los datos faltantes.

Esto es importante porque limpiar textos y tratar valores faltantes son tareas relacionadas, pero no idénticas. Quitar espacios o cambiar mayúsculas puede corregir problemas de escritura, pero no completa información que realmente falta.

Sin embargo, todavía no resolvimos todas las inconsistencias. Por ejemplo, `"Coffee"` y `"COFFEE"` siguen siendo valores diferentes, porque tienen distintas combinaciones de mayúsculas y minúsculas.

Esto muestra una idea importante: la limpieza de textos suele requerir varios pasos. Quitar espacios puede resolver una parte del problema, pero no necesariamente toda la inconsistencia.

## Unificar mayúsculas y minúsculas

Después de quitar espacios innecesarios, todavía puede quedar otro problema: la misma categoría escrita con distintas combinaciones de mayúsculas y minúsculas.

Por ejemplo, estos valores no son iguales para Pandas:

```text
Coffee
COFFEE
coffee
```

Aunque para nosotros probablemente representan el mismo producto, Pandas los interpreta como textos distintos.

Para unificar esta diferencia podemos convertir todos los textos a minúsculas usando `.str.lower()`.

In [107]:
df["producto_limpio"] = df["producto_limpio"].str.lower()

df[["producto", "producto_limpio"]]

,producto,producto_limpio
0,Coffee,coffee
1,coffee,coffee
2,COFFEE,coffee
3,Tea,tea
4,tea,tea
5,Sandwich,sandwich
6,sandwich,sandwich
7,Cake,cake
8,cake,cake
9,Juice,juice


Ahora los valores de `producto_limpio` quedaron en minúsculas.

Esto permite que variantes como `"Coffee"`, `" coffee"` y `"COFFEE"` se unifiquen como `"coffee"`.

Podemos revisar nuevamente las frecuencias de la columna limpia.

In [108]:
df["producto_limpio"].value_counts(dropna=False)

,count
producto_limpio,
coffee,3
tea,2
sandwich,2
cake,2
smoothie,2
juice,1
unknown,1
error,1
None,1


El conteo ahora es más coherente que el original. Las variantes causadas por espacios y mayúsculas quedaron agrupadas.

Sin embargo, todavía aparecen valores que no representan productos reales, como `"unknown"` y `"error"`, además de valores faltantes. Esos casos no se resuelven simplemente con `strip()` o `lower()`, porque no son variantes de escritura de un producto, sino valores problemáticos que requieren una decisión específica.

## Aplicar la misma limpieza a otras columnas

La combinación de `.str.strip()` y `.str.lower()` también puede aplicarse a otras columnas de texto.

En nuestro dataset, `metodo_pago` y `ubicacion` también tienen inconsistencias de escritura. Antes de limpiarlas, revisemos sus valores.

In [109]:
df["metodo_pago"].value_counts(dropna=False)

,count
metodo_pago,
Cash,1
cash,1
CASH,1
Credit Card,1
credit card,1
Credito,1
Crédito,1
Digital Wallet,1
digital wallet,1


In [110]:
df["ubicacion"].value_counts(dropna=False)

,count
ubicacion,
In-store,1
in-store,1
In-store,1
Takeaway,1
takeaway,1
TAKEAWAY,1
Delivery,1
delivery,1
DELIVERY,1


En ambas columnas aparecen valores que deberían unificarse.

Por ejemplo, en `metodo_pago`, valores como `"Cash"`, `"cash"` y `" CASH "` representan la misma forma de pago, pero Pandas los cuenta por separado. En `ubicacion`, ocurre algo similar con `"In-store"`, `"in-store"` y `" In-store "`.

Vamos a crear columnas limpias para conservar los valores originales y poder comparar antes y después.

In [111]:
df["metodo_pago_limpio"] = df["metodo_pago"].str.strip().str.lower()
df["ubicacion_limpia"] = df["ubicacion"].str.strip().str.lower()

df[
    [
        "metodo_pago",
        "metodo_pago_limpio",
        "ubicacion",
        "ubicacion_limpia"
    ]
]

,metodo_pago,metodo_pago_limpio,ubicacion,ubicacion_limpia
0,Cash,cash,In-store,in-store
1,cash,cash,in-store,in-store
2,CASH,cash,In-store,in-store
3,Credit Card,credit card,Takeaway,takeaway
4,credit card,credit card,takeaway,takeaway
5,Credito,credito,TAKEAWAY,takeaway
6,Crédito,crédito,Delivery,delivery
7,Digital Wallet,digital wallet,delivery,delivery
8,digital wallet,digital wallet,DELIVERY,delivery
9,wallet,wallet,In store,in store


Ahora tenemos una versión normalizada de cada columna.

Esta transformación corrige diferencias producidas por espacios al comienzo o al final y por mayúsculas o minúsculas. Sin embargo, todavía pueden quedar variantes más profundas.

Por ejemplo, `"credit card"` y `"credito"` no se unifican con `strip()` y `lower()`, porque son textos distintos. Lo mismo ocurre con `"takeaway"`, `"take away"` y `"take-away"`.

Para resolver esos casos necesitamos una estrategia adicional: reemplazar valores inconsistentes.

## Reemplazar valores inconsistentes

Después de quitar espacios y unificar mayúsculas/minúsculas, algunas categorías todavía pueden quedar separadas.

Esto ocurre cuando los valores no solo difieren en formato, sino también en la forma de escribir la categoría.

Por ejemplo, en la columna `metodo_pago_limpio` podemos encontrar valores como:

```text
credit card
credito
crédito
```

Según el contexto, podríamos decidir que todos esos valores representan pagos con tarjeta de crédito. Del mismo modo, en la columna `ubicacion_limpia`, valores como:

```text
takeaway
take away
take-away
```

podrían representar la misma modalidad de retiro.

Para unificar estos casos podemos usar `replace()`.


In [112]:
df["metodo_pago_limpio"].value_counts(dropna=False)

,count
metodo_pago_limpio,
cash,3
credit card,2
digital wallet,2
debit card,2
crédito,1
credito,1
wallet,1
unknown,1
error,1


In [113]:
df["ubicacion_limpia"].value_counts(dropna=False)

,count
ubicacion_limpia,
in-store,3
takeaway,3
delivery,3
in store,1
unknown,1
error,1
None,1
take away,1
take-away,1


Primero revisamos los valores actuales. Esto es importante porque no deberíamos reemplazar valores sin saber exactamente qué variantes existen.

En este ejemplo vamos a resolver la variante `"credito"` / `"crédito"` mediante reemplazos explícitos. Existen técnicas más generales para normalizar tildes y acentos, pero por ahora nos alcanza con una decisión puntual y visible, porque queremos que cada reemplazo quede claro.

Ahora podemos definir diccionarios de reemplazo. En cada diccionario, las claves son los valores actuales y los valores son las categorías finales que queremos conservar.

In [114]:
reemplazos_pago = {
    "cash": "cash",
    "credit card": "credit_card",
    "credito": "credit_card",
    "crédito": "credit_card",
    "digital wallet": "digital_wallet",
    "wallet": "digital_wallet",
    "debit card": "debit_card",
    "unknown": "unknown",
    "error": "error"
}

reemplazos_ubicacion = {
    "in-store": "in_store",
    "in store": "in_store",
    "takeaway": "takeaway",
    "take away": "takeaway",
    "take-away": "takeaway",
    "delivery": "delivery",
    "unknown": "unknown",
    "error": "error"
}

df["metodo_pago_limpio"] = df["metodo_pago_limpio"].replace(reemplazos_pago)
df["ubicacion_limpia"] = df["ubicacion_limpia"].replace(reemplazos_ubicacion)

Después de aplicar los reemplazos, podemos revisar nuevamente las frecuencias.

In [115]:
df["metodo_pago_limpio"].value_counts(dropna=False)

,count
metodo_pago_limpio,
credit_card,4
cash,3
digital_wallet,3
debit_card,2
unknown,1
error,1
None,1


In [116]:
df["ubicacion_limpia"].value_counts(dropna=False)

,count
ubicacion_limpia,
takeaway,5
in_store,4
delivery,3
unknown,1
error,1
None,1


Ahora las categorías quedaron más unificadas.

El método `replace()` es útil cuando conocemos las variantes que queremos corregir. A diferencia de `strip()` o `lower()`, que aplican una transformación general, `replace()` permite tomar decisiones específicas para cada valor.

Esto también implica una responsabilidad: cada reemplazo debe tener sentido dentro del contexto. No deberíamos unificar categorías solo porque se parecen. Necesitamos saber o decidir razonablemente que representan lo mismo.

Por ejemplo, unificar `"wallet"` con `"digital_wallet"` puede tener sentido si en nuestro criterio ambas expresiones representan la misma forma de pago. Pero esa decisión debería quedar documentada, porque afecta los resultados posteriores.

## Tratar valores desconocidos, errores y faltantes

Después de normalizar espacios, mayúsculas, minúsculas y algunas variantes de escritura, todavía quedan valores que requieren una decisión especial.

En nuestras columnas limpias aparecen valores como:

```text
unknown
error
```

También pueden aparecer valores faltantes reales, representados como `NaN`.

Estos casos no son equivalentes a una categoría común. Por ejemplo, `cash`, `credit_card` o `digital_wallet` representan métodos de pago reales. En cambio, `unknown` indica que el valor no se conoce, y `error` sugiere que hubo algún problema de carga o registro.

Antes de decidir qué hacer, conviene revisar cuántos casos hay de cada tipo.

In [117]:
df["producto_limpio"].value_counts(dropna=False)

,count
producto_limpio,
coffee,3
tea,2
sandwich,2
cake,2
smoothie,2
juice,1
unknown,1
error,1
None,1


In [118]:
df["metodo_pago_limpio"].value_counts(dropna=False)

,count
metodo_pago_limpio,
credit_card,4
cash,3
digital_wallet,3
debit_card,2
unknown,1
error,1
None,1


In [119]:
df["ubicacion_limpia"].value_counts(dropna=False)

,count
ubicacion_limpia,
takeaway,5
in_store,4
delivery,3
unknown,1
error,1
None,1


Una opción posible es conservar `unknown` como una categoría especial. Esto puede ser razonable cuando queremos indicar que el dato no está disponible, pero sin eliminar la fila.

El valor `error`, en cambio, puede requerir otro tratamiento. Podríamos decidir conservarlo como señal de problema, reemplazarlo por `unknown`, convertirlo en faltante o revisar la fila completa antes de tomar una decisión.

En este ejemplo vamos a tomar una decisión simple: vamos a unificar `error` dentro de `unknown`. La idea será tratar ambos casos como valores no confiables o no disponibles para el análisis categórico.

Esta decisión no es universal. En otro contexto, `error` podría analizarse por separado para investigar fallas del sistema de carga. Lo importante es que la decisión sea explícita.

In [120]:
df["producto_limpio"] = df["producto_limpio"].replace({
    "error": "unknown"
})

df["metodo_pago_limpio"] = df["metodo_pago_limpio"].replace({
    "error": "unknown"
})

df["ubicacion_limpia"] = df["ubicacion_limpia"].replace({
    "error": "unknown"
})

Ahora revisamos nuevamente los conteos.

In [121]:
df["producto_limpio"].value_counts(dropna=False)

,count
producto_limpio,
coffee,3
tea,2
sandwich,2
cake,2
unknown,2
smoothie,2
juice,1
None,1


In [122]:
df["metodo_pago_limpio"].value_counts(dropna=False)

,count
metodo_pago_limpio,
credit_card,4
cash,3
digital_wallet,3
unknown,2
debit_card,2
None,1


In [123]:
df["ubicacion_limpia"].value_counts(dropna=False)

,count
ubicacion_limpia,
takeaway,5
in_store,4
delivery,3
unknown,2
None,1


Después del reemplazo, `error` ya no aparece como categoría separada. Sus casos quedaron agrupados dentro de `unknown`.

Todavía pueden quedar valores faltantes reales, como `NaN`. En este capítulo no vamos a profundizar en imputación, porque ya trabajamos ese tema en el tratamiento de valores faltantes. Lo importante aquí es distinguir entre tres situaciones:

```text
categorías válidas
categorías desconocidas o problemáticas
valores faltantes reales
```

Una limpieza de categorías no siempre busca eliminar todos los valores problemáticos. A veces busca hacerlos visibles, agruparlos de manera consistente y evitar que fragmenten el análisis.

## Comparar antes y después de la limpieza

Una buena limpieza de datos debe poder verificarse.

No alcanza con aplicar transformaciones como `.str.strip()`, `.str.lower()` o `replace()`. Después de transformar una columna, necesitamos comprobar si el resultado realmente mejoró el problema que queríamos resolver.

En este caso, podemos comparar los conteos originales con los conteos de las columnas limpias.

Empecemos con `producto`.

In [124]:
print("Conteo original de producto:")
print(df["producto"].value_counts(dropna=False))

print()

print("Conteo de producto_limpio:")
print(df["producto_limpio"].value_counts(dropna=False))

Conteo original de producto:
producto
Coffee       1
 coffee      1
COFFEE       1
Tea          1
tea          1
Sandwich     1
sandwich     1
Cake         1
cake         1
Juice        1
UNKNOWN      1
ERROR        1
None         1
Smoothie     1
smoothie     1
Name: count, dtype: int64

Conteo de producto_limpio:
producto_limpio
coffee      3
tea         2
sandwich    2
cake        2
unknown     2
smoothie    2
juice       1
None        1
Name: count, dtype: int64


La comparación muestra que las variantes de escritura quedaron agrupadas.

Antes, Pandas contaba por separado valores como `"Coffee"`, `" coffee"` y `"COFFEE"`. Después de la limpieza, esas variantes quedaron reunidas bajo una misma categoría: `"coffee"`.

También podemos hacer la misma comparación con el método de pago.

In [125]:
print("Conteo original de metodo_pago:")
print(df["metodo_pago"].value_counts(dropna=False))

print()

print("Conteo de metodo_pago_limpio:")
print(df["metodo_pago_limpio"].value_counts(dropna=False))

Conteo original de metodo_pago:
metodo_pago
Cash              1
cash              1
 CASH             1
Credit Card       1
credit card       1
Credito           1
Crédito           1
Digital Wallet    1
digital wallet    1
wallet            1
UNKNOWN           1
ERROR             1
None              1
Debit Card        1
debit card        1
Name: count, dtype: int64

Conteo de metodo_pago_limpio:
metodo_pago_limpio
credit_card       4
cash              3
digital_wallet    3
unknown           2
debit_card        2
None              1
Name: count, dtype: int64


En este caso, la limpieza no solo corrigió espacios y mayúsculas. También unificó variantes como `"Credit Card"`, `"credit card"`, `"Credito"` y `"Crédito"` bajo una categoría común.

Finalmente, revisemos la ubicación.

In [126]:
print("Conteo original de ubicacion:")
print(df["ubicacion"].value_counts(dropna=False))

print()

print("Conteo de ubicacion_limpia:")
print(df["ubicacion_limpia"].value_counts(dropna=False))

Conteo original de ubicacion:
ubicacion
In-store      1
in-store      1
 In-store     1
Takeaway      1
takeaway      1
TAKEAWAY      1
Delivery      1
delivery      1
DELIVERY      1
In store      1
UNKNOWN       1
ERROR         1
None          1
Take away     1
take-away     1
Name: count, dtype: int64

Conteo de ubicacion_limpia:
ubicacion_limpia
takeaway    5
in_store    4
delivery    3
unknown     2
None        1
Name: count, dtype: int64


Esta comparación permite ver el efecto de la limpieza.

Antes, algunas categorías estaban fragmentadas por diferencias de escritura. Después, las columnas limpias tienen menos variantes y representan mejor las categorías que queremos analizar.

Esto es importante porque los conteos, filtros y agrupamientos posteriores van a depender de estas columnas. Si las categorías quedan fragmentadas, el análisis puede subestimar o sobreestimar ciertos grupos.

Por eso, después de limpiar columnas categóricas, conviene revisar siempre los valores únicos o las frecuencias resultantes.

## Crear una versión limpia del dataset

Hasta ahora fuimos creando columnas limpias junto a las columnas originales.

Esta estrategia tiene una ventaja importante: permite comparar antes y después de la limpieza. Si algo no queda bien, todavía podemos revisar los valores originales y corregir la transformación.

Sin embargo, para continuar un análisis, muchas veces conviene construir una versión más ordenada del dataset. Esa versión puede conservar las columnas limpias y dejar afuera las columnas originales que ya no vamos a usar.

Por ejemplo, podríamos crear un nuevo `DataFrame` llamado `df_limpio`.

In [127]:
df_limpio = df[
    [
        "id_venta",
        "producto_limpio",
        "metodo_pago_limpio",
        "ubicacion_limpia",
        "cantidad",
        "precio_unitario"
    ]
].copy()

df_limpio

,id_venta,producto_limpio,metodo_pago_limpio,ubicacion_limpia,cantidad,precio_unitario
0,V001,coffee,cash,in_store,2,1200
1,V002,coffee,cash,in_store,1,1200
2,V003,coffee,cash,in_store,3,1200
3,V004,tea,credit_card,takeaway,2,900
4,V005,tea,credit_card,takeaway,1,900
5,V006,sandwich,credit_card,takeaway,1,2500
6,V007,sandwich,credit_card,delivery,2,2500
7,V008,cake,digital_wallet,delivery,1,1800
8,V009,cake,digital_wallet,delivery,3,1800
9,V010,juice,digital_wallet,in_store,2,1500


Ahora `df_limpio` contiene solamente las columnas que queremos conservar para el análisis.

Podemos renombrar las columnas limpias para que tengan nombres más simples:

In [128]:
df_limpio = df_limpio.rename(columns={
    "producto_limpio": "producto",
    "metodo_pago_limpio": "metodo_pago",
    "ubicacion_limpia": "ubicacion"
})

df_limpio

,id_venta,producto,metodo_pago,ubicacion,cantidad,precio_unitario
0,V001,coffee,cash,in_store,2,1200
1,V002,coffee,cash,in_store,1,1200
2,V003,coffee,cash,in_store,3,1200
3,V004,tea,credit_card,takeaway,2,900
4,V005,tea,credit_card,takeaway,1,900
5,V006,sandwich,credit_card,takeaway,1,2500
6,V007,sandwich,credit_card,delivery,2,2500
7,V008,cake,digital_wallet,delivery,1,1800
8,V009,cake,digital_wallet,delivery,3,1800
9,V010,juice,digital_wallet,in_store,2,1500


Esta versión del dataset es más clara para trabajar.

La tabla original `df` todavía conserva las columnas originales y las columnas limpias. En cambio, `df_limpio` representa una versión preparada para continuar el análisis.

Esta práctica es recomendable porque separa dos momentos del trabajo:

```text
dataset original o de trabajo → permite revisar y comparar transformaciones
dataset limpio → permite continuar el análisis con columnas consistentes
```

Crear una versión limpia no significa borrar la historia del proceso. Significa construir una tabla más adecuada para seguir trabajando, después de haber verificado las transformaciones realizadas.

## Errores frecuentes al limpiar textos y categorías

Al limpiar columnas de texto, uno de los errores más frecuentes es aplicar transformaciones sin revisar primero los valores existentes.

Por ejemplo, convertir todo a minúsculas o quitar espacios puede ser útil, pero no resuelve todos los problemas. Si una columna contiene valores como `"Credit Card"`, `"Credito"` y `"Crédito"`, aplicar `.str.lower()` no alcanza para unificarlos. Esos valores seguirán siendo distintos hasta que tomemos una decisión explícita de reemplazo.

Otro error común es unificar categorías solo porque se parecen. En algunos casos, dos textos parecidos representan lo mismo, pero en otros pueden tener significados diferentes. Por eso, antes de reemplazar valores conviene revisar el contexto y documentar el criterio usado.

También debemos tener cuidado con valores como `"UNKNOWN"` y `"ERROR"`. No son categorías comunes. Pueden representar ausencia de información, errores de carga o valores que necesitan una revisión especial. Agruparlos dentro de una categoría como `"unknown"` puede ser razonable para ciertos análisis, pero no significa que hayamos recuperado el dato real.

Otro punto importante es no sobrescribir columnas originales demasiado pronto. Durante la limpieza puede ser conveniente crear columnas auxiliares, como `producto_limpio`, `metodo_pago_limpio` o `ubicacion_limpia`, para comparar el antes y el después. Una vez verificada la transformación, podemos construir una versión limpia del dataset.

Finalmente, un error frecuente es no verificar el resultado. Después de limpiar una columna categórica, siempre conviene revisar los valores únicos o los conteos.

In [129]:
df_limpio["producto"].value_counts(dropna=False)

,count
producto,
coffee,3
tea,2
sandwich,2
cake,2
unknown,2
smoothie,2
juice,1
None,1


In [130]:
df_limpio["metodo_pago"].value_counts(dropna=False)

,count
metodo_pago,
credit_card,4
cash,3
digital_wallet,3
unknown,2
debit_card,2
None,1


In [131]:
df_limpio["ubicacion"].value_counts(dropna=False)

,count
ubicacion,
takeaway,5
in_store,4
delivery,3
unknown,2
None,1


Estas verificaciones permiten confirmar que las categorías quedaron más consistentes.

En general, una buena rutina para limpiar textos y categorías podría ser:

```text
revisar valores originales
detectar inconsistencias
aplicar transformaciones generales
reemplazar variantes específicas
decidir qué hacer con valores problemáticos
verificar conteos finales
construir una versión limpia del dataset
```

La limpieza de textos no consiste solo en “prolijar” valores. Su objetivo es evitar que el análisis trate como distintas categorías que en realidad deberían estar agrupadas, o que mezcle categorías válidas con valores desconocidos o erróneos.

## Resumen del capítulo

En este capítulo trabajamos con la limpieza de textos y categorías.

Partimos de un dataset sintético de ventas de cafetería construido especialmente para mostrar problemas frecuentes en columnas categóricas. El objetivo fue observar cómo pequeñas diferencias de escritura pueden afectar el análisis.

Vimos que, para Pandas, valores como `"Coffee"`, `" coffee"` y `"COFFEE"` son distintos. Lo mismo ocurre con textos que tienen espacios al comienzo o al final, diferencias entre mayúsculas y minúsculas, variantes con tildes o distintas formas de escribir una misma categoría.

Primero revisamos los valores originales usando `value_counts(dropna=False)`. Esa revisión nos permitió ver que las categorías estaban fragmentadas:

```python
df["producto"].value_counts(dropna=False)
```

Después empezamos a limpiar la columna `producto`. Usamos `.str.strip()` para quitar espacios innecesarios al comienzo y al final:

```python
df["producto_limpio"] = df["producto"].str.strip()
```

Luego usamos `.str.lower()` para unificar mayúsculas y minúsculas:

```python
df["producto_limpio"] = df["producto_limpio"].str.lower()
```

Esta combinación permitió agrupar variantes como `"Coffee"`, `" coffee"` y `"COFFEE"` bajo una misma categoría.

También aplicamos transformaciones similares en las columnas `metodo_pago` y `ubicacion`, creando versiones limpias de cada una:

```python
df["metodo_pago_limpio"] = df["metodo_pago"].str.strip().str.lower()
df["ubicacion_limpia"] = df["ubicacion"].str.strip().str.lower()
```

Más adelante vimos que algunas inconsistencias no se resuelven solo con espacios y minúsculas. Por ejemplo, valores como `"credit card"`, `"credito"` y `"crédito"` pueden representar la misma forma de pago, pero siguen siendo textos diferentes. Para unificar esos casos usamos `replace()` con diccionarios de reemplazo.

También discutimos cómo tratar valores como `"UNKNOWN"`, `"ERROR"` y valores faltantes reales. Vimos que no son categorías comunes y que requieren una decisión explícita. En este ejemplo decidimos agrupar `"error"` dentro de `"unknown"`, interpretando ambos como valores no confiables o no disponibles para el análisis categórico.

Después comparamos los conteos antes y después de la limpieza. Esta verificación fue importante porque mostró el efecto real de las transformaciones: categorías que antes aparecían fragmentadas quedaron agrupadas de manera más consistente.

Finalmente construimos una versión limpia del dataset:

```python
df_limpio = df[
    [
        "id_venta",
        "producto_limpio",
        "metodo_pago_limpio",
        "ubicacion_limpia",
        "cantidad",
        "precio_unitario"
    ]
].copy()
```

y luego renombramos las columnas limpias para continuar trabajando con nombres más simples.

La idea principal de este capítulo fue:

```text
Limpiar textos y categorías permite evitar que el análisis trate como distintos valores que deberían representar la misma categoría.
```

Esta limpieza no es solo una cuestión de presentación. Afecta directamente conteos, filtros, agrupamientos, gráficos y conclusiones.

## Próximo paso

Ya trabajamos con valores faltantes, duplicados y limpieza de categorías.

El siguiente paso será estudiar la conversión de tipos de datos. En datasets reales, muchas columnas que deberían ser numéricas pueden aparecer cargadas como texto. También puede ocurrir que las fechas no estén interpretadas como fechas, sino como cadenas de caracteres.

Vamos a aprender a revisar tipos de datos, convertir columnas numéricas con control de errores y preparar columnas para que puedan usarse correctamente en cálculos y análisis posteriores.
